# #2 Bulk kinetics and edit frequencies

## Purpose

Determine rate of editing in vivo and in vitro with bulk sequencing of target site amplicons. Also analyze LM distribution to ensure balanced representation.

## Setup

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from devmap.config import get_paths, set_theme, edit_site_palette, edit_ids, edit_site_palette, discrete_colors
from devmap.utils import save_plot
from devmap.plots import barplot_with_points

set_theme()
base_path, plots_path, results_path = get_paths("validation")

## Load data

In [2]:
in_vitro = pd.read_csv(results_path / "in_vitro_bulk.csv")
in_vivo = pd.read_csv(results_path / "in_vivo_bulk.csv")
edit_freq = pd.read_csv(results_path / "in_vivo_bulk_lm_balance.csv")

## Replace site names

In [3]:
site_names = {"RNF2": "ES1", "HEK3": "ES2", "EMX1": "ES3"}
in_vitro["edit site"] = in_vitro["site"].replace(site_names)
in_vitro["edit_pct"] = in_vitro["edit_frac"] * 100
in_vivo["edit site"] = in_vivo["site"].replace(site_names)
in_vivo["edit_pct"] = in_vivo["edit_frac"] * 100

## In vitro time course

In [13]:
import matplotlib as mpl
mpl.rcParams["xtick.major.pad"] = 2
mpl.rcParams["ytick.major.pad"] = 2
mpl.rcParams["legend.title_fontsize"] = 10
fig, axes = plt.subplots(1, 3, figsize=(3, 1.75), sharey=True, layout="constrained")

for ax, speed in zip(axes, ["Slow", "Medium", "Fast"]):
    sns.lineplot(
        data=in_vitro.query("speed == @speed & day <= 9").sort_values("edit site"),
        x="day", y="edit_pct",
        hue="edit site",
        ax=ax,
        palette=edit_site_palette
    )
    ax.set_title(speed)
    if speed == "Medium":
        ax.set_xlabel("Days of editing")
    else:
        ax.set_xlabel("")
    ax.set_ylabel("Mean edit fraction (%)")
    ax.get_legend().remove()
    ax.set_xticks([1, 3, 5, 7, 9])
    ax.set_yticks([0, 20, 40, 60, 80])
    ax.set_ylim(0, 85)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Edit site",
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, -0.25)
)
fig.subplots_adjust(bottom=0.3)
save_plot(plots_path / "in_vitro_bulk_kinetics.svg", fig)

## In vitro kinetics

In [4]:
fig, ax = plt.subplots(figsize=(2, 2.3), layout="constrained")

ax = barplot_with_points(
    data=in_vivo,
    x="speed",
    y="edit_pct",
    hue="edit site",
    order=["Slow", "Medium", "Fast"],
    palette=edit_site_palette,
    ax=ax,
)

ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_xlabel("Editing speed")
ax.set_ylabel("Mean edit fraction (%)")

save_plot(plots_path / "invivo_bulk_kinetics.svg", fig)

## LM balance

In [5]:
def normalized_entropy(distribution):
    """Calculate normalized entropy of a distribution."""
    # Remove zero probabilities for log calculation
    distribution = distribution[distribution > 0]
    # Calculate entropy
    entropy = -np.sum(distribution * np.log2(distribution))
    # Normalize entropy
    num_states = len(distribution)
    normalized_entropy = entropy / np.log2(num_states)
    return normalized_entropy

In [6]:
edit_freq = edit_freq.query("~edit.isin(['*', 'other'])").copy()
edit_freq["read_frac"] = (
    edit_freq.groupby(["site", "sample"])["readCount"]
    .transform(lambda x: x / x.sum())
    * 100
)
fig, axes = plt.subplots(1, 3, figsize=(4, 1.8), layout="constrained", sharey=True)

for i, site in enumerate(["RNF2", "HEK3", "EMX1"]):
    ax = axes[i]
    subset = edit_freq.query("site == @site").copy()
    edit_order = list(edit_ids[site].keys())[1:]
    subset["edit"] = pd.Categorical(
        subset["edit"],
        categories=edit_order,
        ordered=True,
    )

    barplot_with_points(
        data=subset,
        x="edit",
        y="read_frac",
        hue="edit",
        order=edit_order,
        palette=discrete_colors[8],
        ax=ax,
        point_size=3,
        jitter=0.15,
        errorbar="se",
        capsize=0.3,
    )

    probs = subset.groupby("edit", observed=False)["read_frac"].mean().values
    hnorm = normalized_entropy(probs / probs.sum())

    ax.text(
        0.7,
        0.95,
        f"$H_{{norm}} = {hnorm:.2f}$",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=8,
    )

    plt.setp(ax.get_xticklabels(), rotation=90, fontsize=9)
    ax.set_xlabel("")
    ax.set_ylim(0, 40)

axes[0].set_ylabel("Norm. LM installation (%)")

save_plot(plots_path / "lm_balance_barplot.svg", fig)